<a href="https://colab.research.google.com/github/Ali-Hamza-developer/NLP/blob/main/fasttext_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FastText Basics — Full Notes + Installation + Exercises

Easy explanation + code. Covers installation, how FastText differs from Word2Vec, training your own model, pre-trained vectors, handling OOV words, text classification, and saving/loading. One simple diagram only.

## 1. What is FastText? (Easy Explanation)

FastText was built by Facebook (Meta) AI Research. It's very similar to Word2Vec (which you just covered in Gensim) — it also turns words into vectors — but with ONE major upgrade: **it looks at pieces of words (subwords/character n-grams), not just whole words.**

```
Word2Vec sees:   "playing"  -> one single unit, learns ONE vector for it
FastText sees:   "playing"  -> broken into chunks: "pla", "lay", "ayi", "yin", "ing", ...
                              -> the word's vector = combination of all these chunk vectors
```

**Why does this matter?**
1. **Handles typos and rare/misspelled words better.** "playngg" (typo) still shares many subword chunks with "playing", so FastText can still produce a reasonable vector for it.
2. **Handles Out-Of-Vocabulary (OOV) words — a problem Word2Vec/spaCy vectors CANNOT solve.** Recall from earlier notebooks: if Word2Vec/spaCy never saw a word during training, it has NO vector for it at all. FastText can still build a vector for a brand-new word by combining the subword chunks it DOES recognize — even for words it never saw as a whole during training!
3. **Great for morphologically rich languages** (Turkish, Finnish, German, etc.) where words change form a lot (prefixes/suffixes) — subword info captures those patterns.

**Trade-off:** FastText models take a bit more memory and time to train than plain Word2Vec, because it's tracking all those subword chunks too, not just whole words.

## 2. Installation

There are actually TWO ways to use FastText in Python:

**Option A — via Gensim (recommended for this course, same API style as the Word2Vec notebook):**
```
pip install gensim
```
(You likely already have this from the previous Gensim notebook.)

**Option B — Facebook's official `fasttext` library (has extra features like built-in supervised text classification):**
```
pip install fasttext
```
> Note: the official `fasttext` package sometimes needs a C++ compiler present on your system to build from source. If `pip install fasttext` fails, try: `pip install fasttext-wheel` instead (a pre-built version that avoids compiling from source).

This notebook mainly uses **Option A (Gensim)** since it's simpler to set up and keeps the same style as the Word2Vec notebook, with a short section on Option B for supervised text classification.

In [6]:
#!pip install gensim fasttext

In [7]:
import gensim

print(gensim.__version__)


4.4.0


---
## 3. Training Your Own FastText Model

Same training interface as `Word2Vec` from the Gensim notebook — just swap the class name.

In [8]:
from gensim.models import FastText

# a toy corpus - same style as the Word2Vec notebook, small so it's fast to run
sentences = [
    "the king ruled the kingdom wisely".split(),
    "the queen ruled the kingdom wisely".split(),
    "the man walked to the market".split(),
    "the woman walked to the market".split(),
    "the king and queen visited the market".split(),
    "the dog ran across the park".split(),
    "the cat ran across the park".split(),
    "the dog and cat played in the park".split(),
]

# vector_size, window, min_count all mean the same as Word2Vec
# min_n / max_n: the MIN and MAX length of character chunks (subwords) FastText will consider
model = FastText(sentences, vector_size=50, window=3, min_count=1, min_n=2, max_n=4, workers=4)


**Note:** just like the Word2Vec notebook, this toy corpus is FAR too small for genuinely meaningful vectors — this is purely to show the mechanics. Real FastText models need thousands+ of sentences.

In [9]:
model.wv["king"].shape   # 50-number vector, as requested


(50,)

In [10]:
model.wv.most_similar("king")


[('kingdom', 0.45392081141471863),
 ('woman', 0.3424012064933777),
 ('in', 0.25697028636932373),
 ('played', 0.1559862345457077),
 ('walked', 0.14395074546337128),
 ('ruled', 0.12463270872831345),
 ('market', 0.11488242447376251),
 ('queen', 0.09810090065002441),
 ('wisely', 0.09567688405513763),
 ('to', 0.07667029649019241)]

## 4. The Big Difference — FastText Can Handle Words It's NEVER Seen

This is the standout feature. Let's try a word that was NEVER in our training sentences at all.

In [11]:
# "kingdoms" (plural) never appeared in training -- only "kingdom" (singular) did
"kingdoms" in model.wv.key_to_index   # check if it's in the known vocabulary


False

**Output:** `False` — "kingdoms" was never seen as a whole word during training.

In [12]:
# yet FastText can STILL produce a vector for it, using subword chunks it recognizes from "kingdom"
model.wv["kingdoms"].shape


(50,)

In [13]:
model.wv.most_similar("kingdoms")   # even for this unseen word, FastText finds related words


[('kingdom', 0.8178614974021912),
 ('king', 0.41560283303260803),
 ('woman', 0.23505288362503052),
 ('market', 0.15075409412384033),
 ('park', 0.149987131357193),
 ('in', 0.10712318122386932),
 ('queen', 0.10576551407575607),
 ('played', 0.07007013261318207),
 ('dog', 0.06320780515670776),
 ('to', 0.019725777208805084)]

**Compare this to Word2Vec or spaCy's word vectors:** if you tried `model.wv["kingdoms"]` with a Word2Vec model that only ever saw "kingdom", it would throw a `KeyError` — the word simply doesn't exist for it. FastText instead breaks "kingdoms" into character chunks ("king", "ingd", "ngdo", "gdom", "doms"...), recognizes many of those chunks from "kingdom", and builds a reasonable vector anyway. **This solves the Out-Of-Vocabulary problem** that was flagged as a pitfall in the earlier spaCy word-vectors notebook.

---
## 5. Using Pre-Trained FastText Vectors

Just like Gensim's `downloader` gave you pre-trained GloVe/Word2Vec vectors, it also has pre-trained FastText vectors, trained on huge real-world text (Wikipedia + Common Crawl).

In [14]:
import gensim.downloader as api

# see which fasttext-related pre-trained models are available
[name for name in api.info()["models"].keys() if "fasttext" in name.lower()]


['fasttext-wiki-news-subwords-300']

In [ ]:
# download a pre-trained FastText model (this is a sizeable download, requires internet, may take a few minutes)
ft = api.load("fasttext-wiki-news-subwords-300")


[===================-------------------------------] 39.5% 378.1/958.4MB downloaded

In [ ]:
ft.most_similar("computer")


In [ ]:
ft.similarity("king", "queen")


### Testing OOV Handling with the Pre-Trained Model

Let's try a deliberately misspelled/rare word — something almost certainly never seen as a whole token during training.

In [ ]:
"pizzaaaa" in ft.key_to_index   # check if this made-up misspelling exists as a known word


In [ ]:
ft.most_similar("pizzaaaa")   # FastText still gives a sensible answer thanks to subword matching with "pizza"


**This is the killer feature of FastText versus Word2Vec/GloVe** — misspellings, rare inflections, and brand-new words still get usable vectors instead of crashing with a `KeyError` or silently returning nothing.

### Analogy Test — Same as Before

In [ ]:
ft.most_similar(positive=["king", "woman"], negative=["man"], topn=5)


---
## 6. Saving and Loading a Trained FastText Model

In [ ]:
model.save("my_fasttext.model")


In [ ]:
loaded_model = FastText.load("my_fasttext.model")
loaded_model.wv.most_similar("king")


For a downloaded pre-trained `KeyedVectors` object (like `ft` above):
```python
ft.save("fasttext_vectors.kv")
from gensim.models import KeyedVectors
loaded_ft = KeyedVectors.load("fasttext_vectors.kv")
```

---
## 7. Bonus: Facebook's Official `fasttext` Library — Supervised Text Classification

The official `fasttext` package (Option B from the installation section) has a handy trick: it can do fast, simple SUPERVISED text classification directly, without needing sklearn at all — useful for quick baselines.

```python
import fasttext

# fasttext expects a plain text file where each line looks like:
# __label__positive this movie was great
# __label__negative this movie was terrible
model = fasttext.train_supervised(input="train.txt")

# predict on new text
model.predict("this movie was absolutely wonderful")
# -> (('__label__positive',), array([0.98]))
```

This isn't covered hands-on here since it needs a specially-formatted training file, but it's worth knowing this shortcut exists — it's often used as a strong, extremely fast baseline before trying more complex models.

---
## 8. FastText vs Word2Vec vs GloVe — Quick Comparison

| | Word2Vec | GloVe | FastText |
|---|---|---|---|
| Unit learned | whole words | whole words | subword chunks + whole words |
| Handles typos/rare words | No (KeyError if unseen) | No (KeyError if unseen) | **Yes** — builds a vector from subwords |
| Handles brand-new/unseen words | No | No | **Yes** |
| Good for morphologically rich languages | Not great | Not great | **Yes** |
| Training/memory cost | lower | (usually not trained by you — pre-trained only) | slightly higher than Word2Vec |
| Library used in this course | Gensim `Word2Vec` | spaCy's `en_core_web_lg`, or Gensim downloader | Gensim `FastText`, or official `fasttext` package |

**Rule of thumb:** If your text has typos, rare words, slang, or a language with lots of word inflections (prefixes/suffixes), FastText is usually the safer default over plain Word2Vec/GloVe.

---
## Quick Cheat Sheet

| Task | Code |
|---|---|
| Install (gensim route) | `pip install gensim` |
| Install (official route) | `pip install fasttext` (or `fasttext-wheel` if build fails) |
| Train your own FastText model | `FastText(sentences, vector_size=100, window=5, min_count=1, min_n=2, max_n=4)` |
| Get a word's vector | `model.wv["word"]` — works even for unseen words! |
| Check if word was seen during training | `"word" in model.wv.key_to_index` |
| Similar words | `model.wv.most_similar("word")` |
| Word similarity | `model.wv.similarity("w1", "w2")` |
| Analogy (a - b + c) | `model.wv.most_similar(positive=["a","c"], negative=["b"])` |
| Load pre-trained FastText vectors | `gensim.downloader.load("fasttext-wiki-news-subwords-300")` |
| Save / load model | `model.save(path)` / `FastText.load(path)` |
| Official supervised classifier | `fasttext.train_supervised(input="train.txt")` |

---

---
# Practice Exercises (with Solutions)

## Exercise 1 — Train a FastText Model and Test OOV Handling

**Task:** Using the sentences below, train a small FastText model, then test whether it can produce a vector for a word that never appeared in training ("reading" -> test with "readable").

In [ ]:
sentences_ex1 = [
    "she enjoys reading books every evening".split(),
    "he loves reading novels on weekends".split(),
    "reading helps improve vocabulary and focus".split(),
]

model_ex1 = FastText(sentences_ex1, vector_size=30, window=2, min_count=1, min_n=2, max_n=4, workers=1)

print("'readable' in vocabulary:", "readable" in model_ex1.wv.key_to_index)
print("Vector shape for 'readable':", model_ex1.wv["readable"].shape)   # should still work due to subword overlap with "read"/"reading"


**Expected result:** `"readable" in vocabulary` prints `False` (never seen as a whole word), but the vector line still succeeds and returns shape `(30,)` — proving FastText can handle this unseen word via subwords, unlike Word2Vec which would throw a `KeyError`.

## Exercise 2 — Compare FastText vs a Plain Word2Vec on the Same Unseen Word

**Task:** Train BOTH a `Word2Vec` and a `FastText` model on the same tiny corpus, then try to get a vector for an unseen word from both, and observe the difference.

In [ ]:
from gensim.models import Word2Vec

sentences_ex2 = sentences_ex1   # reuse the corpus from Exercise 1

w2v_model = Word2Vec(sentences_ex2, vector_size=30, window=2, min_count=1, workers=1)
ft_model = FastText(sentences_ex2, vector_size=30, window=2, min_count=1, min_n=2, max_n=4, workers=1)

unseen_word = "readability"

# FastText: should succeed
print("FastText vector shape:", ft_model.wv[unseen_word].shape)

# Word2Vec: should fail with a KeyError
try:
    w2v_model.wv[unseen_word]
except KeyError as e:
    print("Word2Vec raised an error as expected:", e)


**Expected result:** FastText successfully returns a `(30,)` vector for "readability" (unseen), while Word2Vec raises a `KeyError` because it has genuinely no information about a word it never saw during training — this is the core practical difference between the two.